## Evaluator-Optimizer Workflow

In this workflow, one LLM call generates a response while another provides evaluation and feedback in a loop.

When to use this workflow
This workflow is particularly effective when we have:

- Clear evaluation criteria
- Value from iterative refinement

The two signs of good fit are:

- LLM responses can be demonstrably improved when feedback is provided
- The LLM can provide meaningful feedback itself


### Improvements from [original cookbook notebook](https://github.com/anthropics/anthropic-cookbook/blob/main/patterns/agents/evaluator_optimizer.ipynb)

- Demonstrates how to use a cheaper model for coding unless it seems to be struggling (frugality).
- It's specifically a code generation evaluator-optimization workflow (yagni):
  - The original implementation tried to be a generalized optimization loop, but it mixed in coding specific prompt language, so it didn't work in a generic context.
  - For a more generalized optimization loop: evaluate could take a criteria variable and the generation should remove code specific references.
  - Adopting a specific workflow means we can easily optimize the prompts for coding.
- Improved the loop so there is no duplication of calls (deduplication).
- Prompts use a single ERB, making it easier to see the full prompt inline (readability).
- Added `<thoughts>` and moved `<evaluation>` to after feedback in the evaluation to trigger CoT and a more objective evaluation (prompt engineering).
- Pass all feedback and only the last attempt. The original passes all attempts and the last feedback (prompt engineering).
- Standardized on not using `[]` when defining the XML respose format (consistency).
- Task no longer includes `<user input>`, it is now a part of the prompt (separating concerns).


## Example

In [1]:
# load required gems and add some helpers for pretty printing in iruby
require_relative "../../../../notebook" 

# a small function for getting xml from responses
def extract_xml(xml_text, tag_name)
  regex = /<#{tag_name}>(.*?)<\/#{tag_name}>/m
  match = xml_text.match(regex)
  match[1].strip if match 
end

# set the default model
Instruct.set_default_model "claude-3-5-sonnet-latest", access_token: ENV['ANTHROPIC_API_KEY'], temperature: 0.1

true

In [3]:
def evaluate_code_for_task(task, code)
    prompt = p.user{"
Evaluate this following code implementation for:
1. code correctness
2. time complexity
3. style and best practices

You should be evaluating only and not attemping to solve the task.
Only output \"PASS\" if all criteria are met and you have no further suggestions for improvements.
Output your evaluation concisely in the following format.

<thoughts>
Your critical evaluation thoughts on the code.
</thoughts>
<feedback>
Concise feedback on what needs improvement and why.
</feedback>
<evaluation>PASS, NEEDS_IMPROVEMENT, or FAIL</evaluation>

Original Task:
<%= task %>

Content to evaluate:
<%= code %>
"} + gen
 result = prompt.call
{ thoughts: extract_xml(result, 'thoughts'), status: extract_xml(result, 'evaluation'), feedback: extract_xml(result, 'feedback') }
end

def generate_attempt(task, previous_attempts, feedback)
    model = previous_attempts.count <= 3 ? "claude-3-haiku-20240307" : "claude-3-5-sonnet-latest"
    prompt = p.user{"Your goal is to complete the task based on <user input>. If there are feedback 
from your previous generations, you should reflect on them to improve your solution

Output your answer concisely in the following format: 

<thoughts>
Your understanding of the task and feedback and how you plan to improve
</thoughts>

<response>
Your code implementation here
</response>
<% unless previous_attempts.empty? %>
Previous Attempts:
    <% previous_attempts.each do | attempt | %>
- <%= attempt %>
    <% end %>
<% end %>
<% unless feedback.nil? %>
Feedback:
<%= feedback %>
<% end %>

Task:
<user input>
<%= task %>
</user input>
"} + gen(model: model, temperature: 0.1)
    result = prompt.call
    { thoughts: extract_xml(result, "thoughts"), code: extract_xml(result, "response") }
end

def generate_code(task)
    attempts = []
    evaluations = []
    loop do
        code_from_previous_attempts = attempts.map{ |a| a[:code] }
        latest_feedback = evaluations.last ? evaluations.last[:feedback] : nil
        attempts << generate_attempt(task, code_from_previous_attempts, latest_feedback)
        puts attempts.last[:code]
        evaluations << evaluate_code_for_task(task, attempts.last[:code])
        evaluation_status = evaluations.last[:status]
        puts "#{evaluation_status}\n#{evaluations.last[:feedback]}"
        break if evaluation_status == "PASS"
    end
    coding_cot = attempts.map{|a| a[:thoughts] }
    evaluations_cot = evaluations.map{|e| e[:thoughts] }
    { cot: coding_cot.zip(evaluations_cot), code: attempts.last[:code] }
end

task = "
Implement a Stack with:
1. push(x)
2. pop()
3. getMin()
All operations should be O(1).
"

output = generate_code(task)
puts output[:cot].join("\n"+"-"*40+"\n")

IRuby::html "<pre><code style='font-family: monospace;'>" + output[:code] + "</code></pre>"

class MinStack:
    def __init__(self):
        self.stack = []
        self.min_stack = []

    def push(self, x: int) -> None:
        self.stack.append(x)
        if not self.min_stack or x <= self.min_stack[-1]:
            self.min_stack.append(x)

    def pop(self) -> None:
        if self.stack[-1] == self.min_stack[-1]:
            self.min_stack.pop()
        self.stack.pop()

    def top(self) -> int:
        return self.stack[-1]

    def getMin(self) -> int:
        return self.min_stack[-1]
NEEDS_IMPROVEMENT
1. Add error handling for empty stack operations
2. Add proper type hints and docstrings
3. Consider adding input validation
4. Consider optimizing space usage by storing (value, count) pairs in min_stack
```python
from typing import Optional

class MinStack:
    """
    A stack data structure that supports push, pop, and getMin operations in O(1) time.
    """
    def __init__(self):
        self.stack: list[int] = []
        self.min_stack: list[tuple[int, int]] = []

"<pre><code style='font-family: monospace;'>```python\nfrom typing import Optional\n\nclass MinStack:\n    \"\"\"\n    A stack data structure that supports push, pop, and getMin operations in O(1) time.\n    \"\"\"\n    def __init__(self):\n        self.stack: list[int] = []\n        self.min_stack: list[tuple[int, int]] = []\n\n    def push(self, x: int) -> None:\n        \"\"\"\n        Pushes the element x onto the stack.\n        \"\"\"\n        self.stack.append(x)\n        if not self.min_stack or x <= self.min_stack[-1][0]:\n            self.min_stack.append((x, 1))\n        else:\n            self.min_stack[-1] = (self.min_stack[-1][0], self.min_stack[-1][1] + 1)\n\n    def pop(self) -> None:\n        \"\"\"\n        Removes the element on the top of the stack.\n        \"\"\"\n        if not self.stack:\n            raise IndexError(\"Cannot pop from an empty stack\")\n\n        if self.stack[-1] == self.min_stack[-1][0]:\n            self.min_stack[-1] = (self.min_stack[-1][0], self.min_stack[-1][1] - 1)\n            if self.min_stack[-1][1] == 0:\n                self.min_stack.pop()\n        self.stack.pop()\n\n    def top(self) -> int:\n        \"\"\"\n        Gets the top element of the stack.\n        \"\"\"\n        if not self.stack:\n            raise IndexError(\"Cannot get the top of an empty stack\")\n        return self.stack[-1]\n\n    def getMin(self) -> int:\n        \"\"\"\n        Retrieves the minimum element in the stack.\n        \"\"\"\n        if not self.min_stack:\n            raise IndexError(\"Cannot get the minimum of an empty stack\")\n        return self.min_stack[-1][0]\n```</code></pre>"